In [ ]:
# 0. Colab 환경 설정: 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
python--version

In [ ]:
# 1. 패키지 설치 (최초 1회)
!pip install google-generativeai sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 78.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [ ]:
# 2. 라이브러리 임포트 및 API 키 세팅
import os, json, faiss, numpy as np
from google import genai
from google.genai import types
from sentence_transformers import SentenceTransformer

client = genai.Client(api_key="YOUR API KEY") # 본 코드는 gemini-2.0-flash 모델 사용


In [ ]:
# 3. 모델, 인덱스, 문서 로드
MODEL_NAME = 'jhgan/ko-sbert-nli'
INDEX_PATH = "/content/drive/MyDrive/RAG/rag_faiss.index"
DOCS_PATH  = "/content/drive/MyDrive/RAG/rag_texts.json"

model = SentenceTransformer(MODEL_NAME)
index = faiss.read_index(INDEX_PATH)
with open(DOCS_PATH, "r", encoding="utf-8") as f:
    docs = json.load(f)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.46k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/538 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# 4. Retriever 클래스 (이전과 동일)
class Retriever:
    def __init__(self, index, docs, model):
        self.index = index
        self.docs  = docs
        self.model = model

    def retrieve(self, query, top_k=5, filters=None):
        q_emb = self.model.encode([query])
        faiss.normalize_L2(q_emb)
        scores, idxs = self.index.search(q_emb, top_k)
        results = []
        for rank, idx in enumerate(idxs[0]):
            results.append({
                "id":       self.docs[idx]["id"],
                "score":    float(scores[0][rank]),
                "context":  self.docs[idx]["context"],
                "metadata": self.docs[idx]["metadata"]
            })
        if filters:
            results = [r for r in results
                       if all(r["metadata"].get(k)==v for k,v in filters.items())][:top_k]
        return results

retriever = Retriever(index=index, docs=docs, model=model)

In [ ]:
def answer_with_rag_gemini(query: str, retriever: Retriever, top_k: int = 5) -> dict:
    # 1) RAG 문서 검색
    docs_top = retriever.retrieve(query=query, top_k=top_k)

    # 2) context 블록 조합
    context_block = "\n\n".join(
        f"{i+1}. {d['context']}" for i, d in enumerate(docs_top)
    )

    # 3) 시스템·사용자 메시지
    system_instr = (
        "당신은 RAG 기반 QA 어시스턴트입니다. 제공된 문서를 참고하여 질문에 답변하세요."
    )
    user_prompt = f"문서:\n{context_block}\n\n질문: {query}\n\n반환 형식은 JSON 객체로, 'answer'와 'sources' 키를 포함해야 합니다."

    # 4) Gemini GenerateContent 호출
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_instr,
            response_mime_type="application/json"
        )
    )
    raw = response.text.strip()
    # (디버깅용) 실제로 받은 raw를 보고 싶으면 주석 해제
    # print("----- RAW RESPONSE -----")
    # print(raw)
    # print("------------------------")

    # 5) 1차 JSON 파싱
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        raise ValueError(f"첫 번째 JSON 파싱 실패:\n{raw}")

    # 6) 만약 parsed가 문자열(내부에 또 JSON이 들어있다면) 한 번 더 파싱
    if isinstance(parsed, str):
        try:
            parsed = json.loads(parsed)
        except json.JSONDecodeError:
            # 두 번째 파싱도 실패하면, 답변만 텍스트로 돌려주도록
            return {"answer": parsed, "sources": []}

    # 7) parsed가 리스트라면, 첫 원소를 사용
    if isinstance(parsed, list):
        if not parsed:
            return {"answer": "", "sources": []}
        parsed = parsed[0]

    # 이제 parsed는 반드시 dict여야 함
    if not isinstance(parsed, dict):
        raise ValueError(f"최종 파싱 결과가 dict가 아님:\n{parsed}")

    # 8) 최종 결과 리턴
    # (없어도 되지만, 키 보장을 위해 default 값을 넣어둡니다)
    return {
        "answer":  parsed.get("answer", ""),
        "sources": parsed.get("sources", [])
    }


In [ ]:
# 6. 사용 예시
query = "이 직무를 선택한 이유는 무엇인가요?"
output = answer_with_rag_gemini(query, retriever, top_k=5)
print(output["answer"])
print(output["sources"])


저는 그동안 현장 일을 많이 뛰었던 경험을 가지고 있습니다. 그렇기 때문에 이 직무 또한 현장의 실무 실무가 굉장히 중요한 역할이고 그리고 그 실무에서 얻는 만족감 성취감 그리고 즐거움 그 모든 거를 이 분야에서 함께 하고 싶어서 지원을 하게 되었습니다. 저는 기존 경험했고 기존에 쌓았던 그런 경험을 통해서 이 업무에서 충분히 발휘할 수 있다 라고 생각을 하고 있고요. 그리고 제가 워낙 적극적이고 긍정적인 마인드를 가지고 일을 할려고 하는 편이라서 새롭게 배우는 이 직무에서 또한 내가 잘 해낼 수 있을 거라 자부할 수 있겠습니다.
['5']


저는 그동안 현장 일을 많이 뛰었던 경험을 가지고 있습니다.   
그렇기 때문에 이 직무 또한 현장의 실무 실무가 굉장히 중요한 역할이고 그리고 그 실무에서 얻는 만족감 성취감   
그리고 즐거움 그 모든 거를 이 분야에서 함께 하고 싶어서 지원을 하게 되었습니다.  
저는 기존 경험했고 기존에 쌓았던 그런 경험을 통해서 이 업무에서 충분히 발휘할 수 있다 라고 생각을 하고 있고요.  
그리고 제가 워낙 적극적이고 긍정적인 마인드를 가지고 일을 할려고 하는 편이라서  
새롭게 배우는 이 직무에서 또한 내가 잘 해낼 수 있을 거라 자부할 수 있겠습니다.

In [ ]:
# 6. 사용 예시
query = "이직을 선택한 이유는 무엇인가요?"
output = answer_with_rag_gemini(query, retriever, top_k=5)
print(output["answer"])
print(output["sources"])


제공된 텍스트에는 이직을 선택한 이유에 대한 직접적인 답변이 없습니다. 문서들은 이직을 고려하는 동료에게 어떤 조언을 해줄 수 있는지에 대한 면접 답변들입니다.
[]
